<a href="https://colab.research.google.com/github/Amazeble/chatterbox-finetuning/blob/main/chatterbox-finetuning_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎙️ Chatterbox TTS Fine-Tuning on Google Colab

This notebook provides a complete workflow for fine-tuning Chatterbox TTS models (Standard and Turbo) with LoRA support.

## Overview
- **Step 1**: Clone repository
- **Step 2**: Connect Google Drive and Copy Dataset
- **Step 3**: Install Dependencies
- **Step 4**: Configure Training Parameters (Select LoRA/Full Fine-tune, Turbo/Non-Turbo)
- **Step 5**: Download & Prepare Models (setup.py)
- **Step 6**: Preprocess & Train Model (Combined - depends on config.py settings from Step 4)
- **Step 7**: Inference (Generate Speech)
- **Step 8**: Merge LoRA Adapter (Optional)


## Step 1: Clone Repository

In [ ]:
#@title 📥 Clone Chatterbox Repository

import os

# Detect environment and set working directory
if os.path.exists('/content'):
    # Running in Google Colab
    WORKSPACE = '/content'
    %cd /content
    print("Detected Google Colab environment")
else:
    # Running locally or in other environments
    WORKSPACE = '/workspace'
    %cd "$WORKSPACE"
    print("Running in local/other environment")

# Clone the repository if it doesn't exist
if not os.path.exists('chatterbox-finetuning'):
    print("Cloning Chatterbox Finetuning repository...")
    !git clone https://github.com/Amazeble/chatterbox-finetuning.git
    %cd chatterbox-finetuning
    print("✅ Repository cloned successfully!")
else:
    print("✅ Repository already exists.")
    %cd chatterbox-finetuning

# Verify contents
print("\nRepository contents:")
!ls -la

## Step 2: Connect Google Drive and Copy Dataset

In [ ]:
#@title 📁 Connect Google Drive and Copy Dataset

import os
from google.colab import drive

# Mount Google Drive
print("Mounting Google Drive...")
drive.mount('/content/drive')

# Define source and destination paths
source_path = '/content/drive/MyDrive/MyTTSDataset/'
dest_path = '/content/chatterbox-finetuning/'

# Check if source path exists
if os.path.exists(source_path):
    print(f"\n✅ Source path found: {source_path}")
    print("Copying dataset to working directory...")

    # Create destination directory if it doesn't exist
    os.makedirs(dest_path, exist_ok=True)

    # Copy files
    !cp -r {source_path}/* {dest_path}

    print(f"✅ Dataset copied successfully to {dest_path}")
    print("\nVerifying copied files:")
    !ls -la {dest_path}
else:
    print(f"⚠️ Source path not found: {source_path}")
    print("Please ensure your dataset is in the correct location in Google Drive.")

## Step 3: Install Dependencies

In [ ]:
#@title 🔧 Install Dependencies

import os

# Install FFmpeg
print("Installing FFmpeg...")
!apt-get update && apt-get install -y ffmpeg

# Install Python dependencies
print("\nInstalling Python dependencies...")
%cd /content/chatterbox-finetuning
!pip install -r requirements.txt

print("\n✅ Dependencies installed successfully!")

## Step 4: Configure Training Parameters

In [ ]:
#@title ⚙️ Configure Training Parameters

#@markdown **Training Options:**
is_turbo = True #@param {type:"boolean"}
is_lora = True #@param {type:"boolean"}

#@markdown **Dataset Format:**
ljspeech = True #@param {type:"boolean"}
json_format = False #@param {type:"boolean"}
preprocess = True #@param {type:"boolean"}

#@markdown **Hyperparameters:**
batch_size = 8 #@param {type:"slider", min:1, max:16, step:1}
learning_rate = 1e-4 #@param {type:"number"}
num_epochs = 10 #@param {type:"slider", min:1, max:50, step:1}
new_vocab_size = 52260 #@param {type:"integer"}

import os
import re
from IPython.display import clear_output

# Check if repository exists
repo_path = '/content/chatterbox-finetuning'
if not os.path.exists(repo_path):
    raise FileNotFoundError(f"❌ Repository not found at {repo_path}! Please run Step 1 (Clone Repository) first.")

%cd /content/chatterbox-finetuning

# Read current config file if it exists, otherwise create boilerplate
if not os.path.exists('src/config.py'):
    os.makedirs('src', exist_ok=True)
    with open('src/config.py', 'w') as f:
        f.write("# Boilerplate\nis_turbo: bool = True\nis_lora: bool = True\nbatch_size: int = 8\nlearning_rate: float = 1e-4\nnum_epochs: int = 10\nljspeech = True\njson_format = False\npreprocess = True\nnew_vocab_size: int = 52260\n")

with open('src/config.py', 'r') as f:
    config_content = f.read()

# Convert booleans to script strings
turbo_val = "True" if is_turbo else "False"
lora_val = "True" if is_lora else "False"
ljspeech_val = "True" if ljspeech else "False"
json_val = "True" if json_format else "False"
preprocess_val = "True" if preprocess else "False"

# Update configurations inside src/config.py using regex
config_content = re.sub(r'is_turbo:\s*bool\s*=\s*.*', f'is_turbo: bool = {turbo_val}', config_content)
config_content = re.sub(r'is_lora:\s*bool\s*=\s*.*', f'is_lora: bool = {lora_val}', config_content)
config_content = re.sub(r'ljspeech\s*=\s*.*', f'ljspeech = {ljspeech_val}', config_content)
config_content = re.sub(r'json_format\s*=\s*.*', f'json_format = {json_val}', config_content)
config_content = re.sub(r'preprocess\s*=\s*.*', f'preprocess = {preprocess_val}', config_content)
config_content = re.sub(r'batch_size:\s*int\s*=\s*.*', f'batch_size: int = {batch_size}', config_content)
config_content = re.sub(r'learning_rate:\s*float\s*=\s*.*', f'learning_rate: float = {learning_rate}', config_content)
config_content = re.sub(r'num_epochs:\s*int\s*=\s*.*', f'num_epochs: int = {num_epochs}', config_content)
config_content = re.sub(r'new_vocab_size:\s*int\s*=\s*.*', f'new_vocab_size: int = {new_vocab_size}', config_content)

# Save the updated configurations
with open('src/config.py', 'w') as f:
    f.write(config_content)

# Show confirmation of written parameters
clear_output()
print("="*50)
print("✅ Applied settings to src/config.py:")
print("="*50)
print(f"  - is_turbo={turbo_val}")
print(f"  - is_lora={lora_val}")
print(f"  - ljspeech={ljspeech_val}")
print(f"  - json_format={json_val}")
print(f"  - preprocess={preprocess_val}")
print(f"  - batch_size={batch_size}")
print(f"  - learning_rate={learning_rate}")
print(f"  - num_epochs={num_epochs}")
print(f"  - new_vocab_size={new_vocab_size}")
print("\n⚠️ IMPORTANT: Now proceed to Step 5 to run setup.py!")


In [ ]:
#@title 📥 Download & Prepare Models (setup.py)

%cd /content/chatterbox-finetuning

print("Running setup.py to download models...")
print("This may take several minutes depending on your internet connection.")

# Run setup.py
!python setup.py

print("\n✅ Setup completed!")
print("Check the output above for the vocab_size if using Turbo mode.")
print("You may need to update new_vocab_size in src/config.py with the value from setup.py output.")

In [ ]:
#@title 🔄 Preprocess & Train Model
#@markdown This combined cell handles both preprocessing and training based on your config.py settings.
#@markdown **Preprocessing** depends on the settings from Step 4 (config.py):
#@markdown - `preprocess=True`: Will preprocess dataset before training
#@markdown - `preprocess=False`: Will skip preprocessing and go straight to training
#@markdown
#@markdown **Training** will use these settings from config.py:
#@markdown - `is_turbo`: Turbo mode for faster training
#@markdown - `is_lora`: LoRA fine-tuning (recommended for <10h data)
#@markdown - `batch_size`, `learning_rate`, `num_epochs`: Training hyperparameters

%cd /content/chatterbox-finetuning

print("="*50)
print("🔄 Preprocess & Train Model")
print("="*50)

# Read config to show current settings
with open('src/config.py', 'r') as f:
    config_content = f.read()

print("\n📋 Current Configuration:")
print("-" * 50)

# Extract key settings
import re
settings = {
    'Turbo Mode': r'is_turbo:\s*bool\s*=\s*(\w+)',
    'LoRA Training': r'is_lora:\s*bool\s*=\s*(\w+)',
    'Batch Size': r'batch_size:\s*int\s*=\s*(\d+)',
    'Learning Rate': r'learning_rate:\s*float\s*=\s*([\d.e+-]+)',
    'Epochs': r'num_epochs:\s*int\s*=\s*(\d+)',
    'Preprocess': r'preprocess\s*=\s*(\w+)',
    'LJSpeech Format': r'ljspeech\s*=\s*(\w+)',
    'JSON Format': r'json_format\s*=\s*(\w+)'
}

for name, pattern in settings.items():
    match = re.search(pattern, config_content)
    if match:
        print(f"{name}: {match.group(1)}")

print("-" * 50)

if 'preprocess=True' in config_content or 'preprocess = True' in config_content:
    print("\n⏳ Starting preprocessing (this may take several minutes)...")
    !python train.py
    print("\n✅ Preprocessing completed!")
    print("\n" + "="*50)
    print("🏋️ Starting training...")
    print("="*50)
    print("Training progress and audio samples will be displayed below.")
    print("Press Ctrl+C to stop training early.")
    !python train.py
else:
    print("\n⏳ Skipping preprocessing (preprocess=False in config.py)")
    print("\n🏋️ Starting training...")
    print("="*50)
    print("Training progress and audio samples will be displayed below.")
    print("Press Ctrl+C to stop training early.")
    !python train.py

print("\n✅ Training completed!")
print("Check the chatterbox_output/ directory for your trained model.")

In [ ]:
#@title 🗣️ Inference - Generate Speech

%cd /content/chatterbox-finetuning

# First, let's check what models are available
print("Checking available models...")
!ls -la chatterbox_output/

print("\n" + "="*50)
print("To generate speech, edit inference.py with your desired text:")
print("  TEXT_TO_SAY = \"Your text here\"")
print("  AUDIO_PROMPT = \"./speaker_reference/reference.wav\"")
print("="*50)

# Run inference
print("\nRunning inference...")
!python inference.py

print("\n✅ Speech generated!")
print("Output saved as output.wav")

# Play the generated audio
from IPython.display import Audio, display
print("\nPlaying generated audio:")
display(Audio('./output.wav'))

In [ ]:
#@title 📦 Merge LoRA Adapter (Optional)

%cd /content/chatterbox-finetuning

print("Merging LoRA adapter into base model...")
print("This creates a standalone .safetensors file that doesn't require LoRA adapters.")

!python merge_lora.py

print("\n✅ Merge completed!")
print("Your merged model is ready for production use.")
print("Check chatterbox_output/ for the merged .safetensors file.")